# Tutorial: Exploration, Appendix Lines, BUG, and Readiness Tracks

**Audience**
- Readers who want the near-full active modeling picture without reading the entire extension pipeline.

**Prerequisites**
- The first two notebooks.
- Basic familiarity with robustness checks and ablation studies.

**Learning goals**
- Understand how `03_exploration_pipeline.py` groups multiple modeling questions.
- Separate real modeling branches from helper code.
- See which extension lines improved ranking, which failed, and why BUG2 is different.


## Outline

1. The shared evaluation shell.
2. Quality-adjusted and hazard-aware transport.
3. BUG proxy and BUG detectability tracks.
4. BUG2 official-inventory validation.
5. Readiness scoring and event expansion.


In [ ]:
from __future__ import annotations

import ast
import json
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
MODELING_DIR = REPO_ROOT / "project" / "modeling"
OUTPUT_DIR = MODELING_DIR / "output"


def load_json(rel_path: str):
    return json.loads((REPO_ROOT / rel_path).read_text(encoding="utf-8"))


def load_csv(rel_path: str) -> pd.DataFrame:
    return pd.read_csv(REPO_ROOT / rel_path)


def get_def_source(rel_path: str, name: str, max_lines: int = 80) -> str:
    source = (REPO_ROOT / rel_path).read_text(encoding="utf-8")
    tree = ast.parse(source)
    lines = source.splitlines()
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == name:
            block = lines[node.lineno - 1 : node.end_lineno]
            if len(block) > max_lines:
                block = block[:max_lines] + ["# ... truncated for notebook readability ..."]
            return "\n".join(block)
    raise KeyError(f"{name} was not found in {rel_path}")


def print_defs(rel_path: str, *names: str, max_lines: int = 80) -> None:
    for name in names:
        print(f"\n===== {name} ({rel_path}) =====\n")
        print(get_def_source(rel_path, name, max_lines=max_lines))


print("Repository root:", REPO_ROOT)
print("Modeling directory:", MODELING_DIR)


## How to read this giant file

`03_exploration_pipeline.py` contains:

- shared evaluation logic
- several concrete modeling branches
- report/plot writers
- acquisition helpers for future event expansion

This notebook focuses on the modeling branches, not the serialization plumbing.


In [ ]:
quality = load_csv('project/modeling/output/quality_transport_aggregate_metrics_v1.csv')
hazard = load_csv('project/modeling/output/hazard_transport_aggregate_metrics_v1.csv')
bug = load_csv('project/modeling/output/bug_transport_aggregate_metrics_v1.csv')
detect = load_csv('project/modeling/output/bug_detectability_transport_aggregate_metrics_v1.csv')
ready = load_csv('project/modeling/output/event_readiness_score_v1.csv')
qa = load_csv('project/modeling/output/bug2_pr_pilot_qa_v1.csv')

display(quality)
display(hazard)
display(bug)
display(detect)
display(ready)
display(qa)


## 1. The shared evaluation shell

### `project/modeling/pipelines/03_exploration_pipeline.py::run_loeo_spec`

```python
def run_loeo_spec(
    panel: pd.DataFrame,
    recovery: pd.DataFrame,
    numeric_terms: Sequence[str],
    cat_terms: Sequence[str],
    experiment_family: str,
    spec_id: str,
    allowed_events: Optional[Sequence[str]] = None,
) -> SpecResult:
    df = panel.copy()
    rec = recovery.copy()

    if allowed_events is not None:
        allowed = set(allowed_events)
        df = df[df["event_id"].isin(allowed)].copy()
        rec = rec[rec["event_id"].isin(allowed)].copy()

    events = sorted(df["event_id"].dropna().unique().tolist())
    fold_rows: List[Dict[str, object]] = []
    coef_rows: List[Dict[str, object]] = []

    for fold_event in events:
        tr = df[df["event_id"] != fold_event].copy()
        te = df[df["event_id"] == fold_event].copy()
        tr_rec = rec[rec["event_id"] != fold_event].copy()
        te_rec = rec[rec["event_id"] == fold_event].copy()

        tr = _prepare_columns(tr, numeric_terms, cat_terms, required_numeric=("delta_ntl", "is_damaged"))
        te = _prepare_columns(te, numeric_terms, cat_terms, required_numeric=("delta_ntl", "is_damaged"))
        tr_rec = _prepare_columns(tr_rec, numeric_terms, cat_terms, required_numeric=("recovery_days", "event_observed"))
        te_rec = _prepare_columns(te_rec, numeric_terms, cat_terms, required_numeric=("recovery_days", "event_observed"))

        rows, coefs = _evaluate_fold(
            tr,
            te,
            tr_rec,
            te_rec,
            numeric_terms=numeric_terms,
            cat_terms=cat_terms,
            experiment_family=experiment_family,
            spec_id=spec_id,
            fold_event=fold_event,
        )
        fold_rows.extend(rows)
        coef_rows.extend(coefs)

    fold_df = pd.DataFrame(fold_rows)
    agg_df = _aggregate_metrics(fold_df)
    coef_df = pd.DataFrame(coef_rows)
    return SpecResult(fold_df=fold_df, agg_df=agg_df, coef_df=coef_df)
```

## 2. Quality-adjusted and hazard-aware transport

### `project/modeling/pipelines/03_exploration_pipeline.py::build_target_quality_panel`

```python
def build_target_quality_panel(panel: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    defaults = load_json(CONFIG_DEFAULTS)
    recovery_thr = float(defaults["recovery_threshold"])
    damage_thr = float(defaults["damage_threshold"])
    events_cfg = load_json(CONFIG_EVENTS)

    out = panel.copy()
    out["delta_ntl_raw"] = out["delta_ntl"]
    out["valid_pre_days"] = np.nan
    out["valid_post_days"] = np.nan
    out["post_obs_span_days"] = np.nan
    out["recovery_day_first_valid"] = np.nan
    out["recovery_day_first_threshold_hit"] = np.nan
    out["recovery_obs_quality_score"] = np.nan
    out["high_censoring_risk_flag"] = 0
    out["delta_ntl_obs_adjusted"] = out["delta_ntl"]

    rec_parts: List[pd.DataFrame] = []
    audit_rows: List[Dict[str, object]] = []

    for event_id, cfg in events_cfg.items():
        mask = out["event_id"] == event_id
        if mask.sum() == 0:
            continue
        sub = out.loc[mask].copy()
        pre_paths = list_daily_tifs(ROOT / cfg["pre_dir"])
        post_paths = list_daily_tifs(ROOT / cfg["post_dir"])
        if not pre_paths or not post_paths:
            continue

        pre_vals = _extract_pixel_stack_values(pre_paths, sub)
        post_vals = _extract_pixel_stack_values(post_paths, sub)

        valid_pre_days = np.isfinite(pre_vals).sum(axis=1).astype(float)
        valid_post_days = np.isfinite(post_vals).sum(axis=1).astype(float)
        total_pre = float(pre_vals.shape[1]) if pre_vals.ndim == 2 else 0.0
        total_post = float(post_vals.shape[1]) if post_vals.ndim == 2 else 0.0

        pre_ratio = np.divide(valid_pre_days, total_pre, out=np.zeros_like(valid_pre_days), where=total_pre > 0)
        post_ratio = np.divide(valid_post_days, total_post, out=np.zeros_like(valid_post_days), where=total_post > 0)
        obs_weight = np.sqrt(np.clip(pre_ratio, 0.0, 1.0) * np.clip(post_ratio, 0.0, 1.0))

        event_ref = float(sub.loc[obs_weight >= 0.8, "delta_ntl"].median()) if np.any(obs_weight >= 0.8) else float(sub["delta_ntl"].median())
        delta_adj = obs_weight * sub["delta_ntl"].to_numpy() + (1.0 - obs_weight) * event_ref

        targets = sub["pre_mean_ntl"].to_numpy(dtype=float) * recovery_thr
        valid_mask = np.isfinite(post_vals)
        hit_mask = valid_mask & (post_vals >= targets[:, None])
        observed = hit_mask.any(axis=1).astype(int)
        first_valid = np.where(valid_mask.any(axis=1), valid_mask.argmax(axis=1) + 1, np.ceil(total_post).astype(int))
        first_hit = np.where(observed == 1, hit_mask.argmax(axis=1) + 1, np.ceil(total_post).astype(int))
        quality_score = np.divide(valid_post_days, total_post, out=np.zeros_like(valid_post_days), where=total_post > 0)
        high_censor = ((quality_score < 0.60) | ((observed == 0) & (quality_score < 0.85))).astype(int)

        out.loc[mask, "valid_pre_days"] = valid_pre_days
        out.loc[mask, "valid_post_days"] = valid_post_days
        out.loc[mask, "post_obs_span_days"] = total_post
        out.loc[mask, "recovery_day_first_valid"] = first_valid
        out.loc[mask, "recovery_day_first_threshold_hit"] = first_hit
        out.loc[mask, "recovery_obs_quality_score"] = quality_score
        out.loc[mask, "high_censoring_risk_flag"] = high_censor
        out.loc[mask, "delta_ntl_obs_adjusted"] = delta_adj

        event_rec = sub[["pixel_id", "event_id"]].copy()
        event_rec["recovery_days"] = first_hit.astype(int)
        event_rec["event_observed"] = observed.astype(int)
        event_rec["recovery_threshold"] = recovery_thr
        event_rec["valid_pre_days"] = valid_pre_days
        event_rec["valid_post_days"] = valid_post_days
        event_rec["post_obs_span_days"] = total_post
        event_rec["recovery_day_first_valid"] = first_valid
        event_rec["recovery_day_first_threshold_hit"] = first_hit
        event_rec["recovery_obs_quality_score"] = quality_score
        event_rec["high_censoring_risk_flag"] = high_censor
        rec_parts.append(event_rec)

        audit_rows.append(
            {
                "event_id": event_id,
                "n_obs": int(len(sub)),
                "total_pre_days": int(total_pre),
                "total_post_days": int(total_post),
                "mean_pre_valid_ratio_raw": float(pre_ratio.mean()),
                "mean_post_valid_ratio_raw": float(post_ratio.mean()),
                "mean_obs_weight": float(obs_weight.mean()),
                "event_ref_delta": event_ref,
                "delta_shift_mean": float(np.mean(delta_adj - sub["delta_ntl"].to_numpy())),
                "delta_shift_abs_mean": float(np.mean(np.abs(delta_adj - sub["delta_ntl"].to_numpy()))),
                "observed_rate_v2": float(observed.mean()),
                "high_censoring_share": float(high_censor.mean()),
            }
        )

    out["is_damaged_raw"] = out["is_damaged"]
    out["is_damaged"] = (out["delta_ntl_obs_adjusted"] < damage_thr).astype(int)
    out["delta_ntl"] = out["delta_ntl_obs_adjusted"]
    out["pixel_pre_valid_ratio"] = _safe_numeric(out["pixel_pre_valid_ratio"])
    out["pixel_post_valid_ratio"] = _safe_numeric(out["pixel_post_valid_ratio"])
    out["recovery_obs_quality_score"] = _safe_numeric(out["recovery_obs_quality_score"])
    out["high_censoring_risk_flag"] = _safe_numeric(out["high_censoring_risk_flag"]).astype(int)

    rec = pd.concat(rec_parts, ignore_index=True)
    merge_cols = [
        "pixel_id",
        "event_id",
        "in_buffer",
        "pre_mean_ntl",
        "land_use_group",
        "event_disaster_type",
        "osm_dist_any_m",
# ... truncated for notebook readability (19 hidden lines) ...
```

### `project/modeling/pipelines/03_exploration_pipeline.py::run_quality_transport`

```python
def run_quality_transport(panel: pd.DataFrame, recovery: pd.DataFrame) -> SpecResult:
    numeric_terms = BASE_NUMERIC + [
        "pixel_cloud_proxy",
        "pixel_pre_valid_ratio",
        "pixel_post_valid_ratio",
        "recovery_obs_quality_score",
        "high_censoring_risk_flag",
    ]
    cat_terms = ["land_use_group", "event_disaster_type"]
    res = run_loeo_spec(
        panel,
        recovery,
        numeric_terms=numeric_terms,
        cat_terms=cat_terms,
        experiment_family="quality_transport",
        spec_id="QT1",
    )
    res.fold_df.to_csv(QUALITY_TRANSPORT_FOLD_PATH, index=False)
    res.agg_df.to_csv(QUALITY_TRANSPORT_AGG_PATH, index=False)
    return res
```

### `project/modeling/pipelines/03_exploration_pipeline.py::attach_hazard_exposure_features`

```python
def attach_hazard_exposure_features(panel: pd.DataFrame) -> pd.DataFrame:
    out = panel.copy()
    if not EVENT_PROFILE_V1_PATH.exists():
        raise FileNotFoundError(f"Missing event profile: {EVENT_PROFILE_V1_PATH}")

    prof = pd.read_csv(EVENT_PROFILE_V1_PATH).copy()
    prof["event_disaster_type"] = prof["disaster_type"].fillna("unknown").astype(str)
    prof["event_island_like_flag"] = _safe_numeric(prof["island_like_flag"])
    prof["event_cloud_shift"] = _safe_numeric(prof["cloud_post_event_mean"]) - _safe_numeric(prof["cloud_pre_event_mean"])
    prof["event_precip_log1p"] = np.log1p(_safe_numeric(prof["storm_precip_7d"]).clip(lower=0))
    prof["event_duration_log1p"] = np.log1p(_safe_numeric(prof["event_duration_days"]).clip(lower=0))
    prof["event_elevation_log1p"] = np.log1p(_safe_numeric(prof["elevation_median"]).clip(lower=0))
    prof["event_slope_milli"] = _safe_numeric(prof["slope_median"]).clip(lower=0) * 1000.0
    prof["event_urban_context"] = _safe_numeric(prof["urban_share_1km"])
    prof["event_water_context"] = _safe_numeric(prof["water_share_1km"])

    keep = [
        "event_id",
        "event_disaster_type",
        "event_island_like_flag",
        "event_cloud_shift",
        "event_precip_log1p",
        "event_duration_log1p",
        "event_elevation_log1p",
        "event_slope_milli",
        "event_urban_context",
        "event_water_context",
    ]
    out = out.drop(columns=[c for c in keep if c in out.columns and c != "event_id"], errors="ignore")
    out = out.merge(prof[keep], on="event_id", how="left")

    out["event_disaster_type"] = out["event_disaster_type"].fillna("unknown").astype(str)
    for c in [
        "event_island_like_flag",
        "event_cloud_shift",
        "event_precip_log1p",
        "event_duration_log1p",
        "event_elevation_log1p",
        "event_slope_milli",
        "event_urban_context",
        "event_water_context",
    ]:
        out[c] = _safe_numeric(out[c])

    out["island_local_water"] = out["event_island_like_flag"] * _safe_numeric(out["water_share_1km"])
    out["island_local_urban"] = out["event_island_like_flag"] * _safe_numeric(out["urban_share_1km"])
    out["hazard_cloud_water"] = out["event_cloud_shift"] * _safe_numeric(out["water_share_1km"])
    out["hazard_precip_urban"] = out["event_precip_log1p"] * _safe_numeric(out["urban_share_1km"])
    out.to_parquet(PANEL_HAZARD_PATH, index=False)
    return out
```

### `project/modeling/pipelines/03_exploration_pipeline.py::run_hazard_aware_transport`

```python
def run_hazard_aware_transport(
    panel: pd.DataFrame,
    recovery: pd.DataFrame,
    spec_id: str = "HZ1",
    experiment_family: str = "hazard_transport",
    fold_path: Path = HAZARD_TRANSPORT_FOLD_PATH,
    agg_path: Path = HAZARD_TRANSPORT_AGG_PATH,
) -> SpecResult:
    numeric_terms = BASE_NUMERIC + [
        "pixel_cloud_proxy",
        "recovery_obs_quality_score",
        "event_cloud_shift",
        "event_precip_log1p",
        "event_duration_log1p",
        "event_elevation_log1p",
        "event_slope_milli",
        "island_local_water",
        "island_local_urban",
        "hazard_cloud_water",
        "hazard_precip_urban",
    ]
    cat_terms = ["land_use_group", "event_disaster_type"]
    res = run_loeo_spec(
        panel,
        recovery,
        numeric_terms=numeric_terms,
        cat_terms=cat_terms,
        experiment_family=experiment_family,
        spec_id=spec_id,
    )
    res.fold_df.to_csv(fold_path, index=False)
    res.agg_df.to_csv(agg_path, index=False)
    return res
```

### `project/modeling/pipelines/03_exploration_pipeline.py::build_event_selection_scorecard`

```python
def build_event_selection_scorecard(
    hazard_fold: pd.DataFrame,
    target_audit: pd.DataFrame,
) -> pd.DataFrame:
    prof = pd.read_csv(EVENT_PROFILE_V1_PATH).copy()
    shift = pd.read_csv(SHIFT_V3_PATH) if SHIFT_V3_PATH.exists() else pd.DataFrame()

    logit = hazard_fold[hazard_fold["model"] == "Logit"][["fold_event", "auc", "brier"]].rename(
        columns={"fold_event": "event_id", "auc": "logit_auc_hz", "brier": "logit_brier_hz"}
    )
    cox = hazard_fold[hazard_fold["model"] == "Cox"][["fold_event", "c_index"]].rename(
        columns={"fold_event": "event_id", "c_index": "cox_c_index_hz"}
    )
    aft = hazard_fold[hazard_fold["model"] == "AFT"][["fold_event", "c_index"]].rename(
        columns={"fold_event": "event_id", "c_index": "aft_c_index_hz"}
    )

    out = prof.merge(logit, on="event_id", how="left").merge(cox, on="event_id", how="left").merge(aft, on="event_id", how="left")
    out["survival_best_hz"] = out[["cox_c_index_hz", "aft_c_index_hz"]].max(axis=1)
    if not target_audit.empty:
        out = out.merge(target_audit[["event_id", "observed_rate_v2", "high_censoring_share"]], on="event_id", how="left")
    if not shift.empty and "event_id" in shift.columns:
        shift_keep = [c for c in ["event_id", "smd_mean", "psi_mean"] if c in shift.columns]
        shift_small = shift[shift_keep].copy()
        shift_small = shift_small.groupby("event_id", as_index=False).mean(numeric_only=True)
        out = out.merge(shift_small, on="event_id", how="left")

    out["urban_bin"] = pd.cut(
        _safe_numeric(out["urban_share_1km"]),
        bins=[-np.inf, 0.68, 0.75, np.inf],
        labels=["low_urban", "mid_urban", "high_urban"],
    ).astype(str)
    out["water_bin"] = pd.cut(
        _safe_numeric(out["water_share_1km"]),
        bins=[-np.inf, 0.10, 0.15, np.inf],
        labels=["low_water", "mid_water", "high_water"],
    ).astype(str)

    disaster_counts = out["disaster_type"].value_counts(dropna=False).to_dict()
    urban_counts = out["urban_bin"].value_counts(dropna=False).to_dict()
    island_counts = out["island_like_flag"].value_counts(dropna=False).to_dict()

    recs = []
    for row in out.itertuples(index=False):
        reasons: List[str] = []
        if disaster_counts.get(row.disaster_type, 0) <= 1:
            reasons.append(f"add_more_{row.disaster_type}")
        if row.island_like_flag == 1 and island_counts.get(1, 0) <= 2:
            reasons.append("add_non_sanjuan_island_like_event")
        if urban_counts.get(row.urban_bin, 0) <= 1:
            reasons.append(f"add_more_{row.urban_bin}_events")
        if pd.notna(row.logit_auc_hz) and row.logit_auc_hz < 0.45:
            reasons.append("poor_damage_transport_holdout")
        if pd.notna(row.survival_best_hz) and row.survival_best_hz < 0.50:
            reasons.append("poor_recovery_transport_holdout")
        if pd.notna(getattr(row, "observed_rate_v2", np.nan)) and row.observed_rate_v2 < 0.80:
            reasons.append("low_observation_quality_neighbor_needed")
        recs.append(";".join(reasons) if reasons else "representative_keep")

    out["selection_signal"] = recs
    keep_cols = [
        "event_id",
        "disaster_type",
        "island_like_flag",
        "urban_bin",
        "water_bin",
        "storm_precip_7d",
        "event_duration_days",
        "logit_auc_hz",
        "survival_best_hz",
        "observed_rate_v2",
        "smd_mean",
        "psi_mean",
        "selection_signal",
    ]
    keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[keep_cols].drop_duplicates(subset=["event_id"]).sort_values(["logit_auc_hz", "survival_best_hz"], na_position="last")
    out.to_csv(EVENT_SELECTION_PATH, index=False)
    return out
```

## 3. BUG proxy and BUG detectability tracks

### `project/modeling/pipelines/03_exploration_pipeline.py::attach_bug_prior_features`

```python
def attach_bug_prior_features(panel: pd.DataFrame, write_artifacts: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    out = panel.copy()
    events_cfg = load_json(CONFIG_EVENTS)
    lookup = _load_bug_prior_lookup()
    default_row = {
        "bug_propensity_weight": 0.15,
        "night_use_weight": 0.25,
        "detectability_weight": 0.25,
        "capacity_prior_band": "low",
    }
    if BUG_PRIOR_CONFIG_PATH.exists():
        cfg = load_json(BUG_PRIOR_CONFIG_PATH)
        if isinstance(cfg, dict) and isinstance(cfg.get("default"), dict):
            default_row.update(cfg["default"])

    feature_defaults = {
        "bug_prior_count_750m": 0.0,
        "bug_prior_count_1250m": 0.0,
        "bug_prior_capacity_proxy_1km": 0.0,
        "bug_prior_hours_proxy_1km": 0.0,
        "bug_prior_min_dist_m": np.nan,
        "high_conf_bug_buffer": 0,
    }
    for col, default in feature_defaults.items():
        if col not in out.columns:
            out[col] = default

    audit_rows: List[Dict[str, object]] = []

    for event_id, cfg in events_cfg.items():
        mask = out["event_id"].astype(str) == str(event_id)
        if mask.sum() == 0:
            continue

        poi_path = ROOT / str(cfg["poi_csv"])
        if not poi_path.exists():
            out.loc[mask, "bug_prior_count_750m"] = 0.0
            out.loc[mask, "bug_prior_count_1250m"] = 0.0
            out.loc[mask, "bug_prior_capacity_proxy_1km"] = 0.0
            out.loc[mask, "bug_prior_hours_proxy_1km"] = 0.0
            out.loc[mask, "bug_prior_min_dist_m"] = np.nan
            out.loc[mask, "high_conf_bug_buffer"] = 0
            audit_rows.append(
                {
                    "event_id": event_id,
                    "n_obs": int(mask.sum()),
                    "poi_status": "missing_poi_csv",
                    "poi_count": 0,
                    "high_conf_poi_count": 0,
                    "high_conf_share": 0.0,
                    "bug_prior_count_750m_constant": 1,
                    "bug_prior_count_1250m_constant": 1,
                    "bug_prior_capacity_proxy_1km_constant": 1,
                    "bug_prior_hours_proxy_1km_constant": 1,
                    "high_conf_bug_buffer_share": 0.0,
                }
            )
            continue

        poi = pd.read_csv(poi_path).copy()
        if poi.empty:
            out.loc[mask, "bug_prior_count_750m"] = 0.0
            out.loc[mask, "bug_prior_count_1250m"] = 0.0
            out.loc[mask, "bug_prior_capacity_proxy_1km"] = 0.0
            out.loc[mask, "bug_prior_hours_proxy_1km"] = 0.0
            out.loc[mask, "bug_prior_min_dist_m"] = np.nan
            out.loc[mask, "high_conf_bug_buffer"] = 0
            audit_rows.append(
                {
                    "event_id": event_id,
                    "n_obs": int(mask.sum()),
                    "poi_status": "empty_poi_csv",
                    "poi_count": 0,
                    "high_conf_poi_count": 0,
                    "high_conf_share": 0.0,
                    "bug_prior_count_750m_constant": 1,
                    "bug_prior_count_1250m_constant": 1,
                    "bug_prior_capacity_proxy_1km_constant": 1,
                    "bug_prior_hours_proxy_1km_constant": 1,
                    "high_conf_bug_buffer_share": 0.0,
                }
            )
            continue

        poi["facility_type_std"] = poi["type"].astype(str).map(standardize_facility_type)
        poi = poi.merge(lookup, on="facility_type_std", how="left")
        for c, default in [
            ("bug_propensity_weight", float(default_row["bug_propensity_weight"])),
            ("night_use_weight", float(default_row["night_use_weight"])),
            ("detectability_weight", float(default_row["detectability_weight"])),
        ]:
            poi[c] = _safe_numeric(poi[c], default=default)
        poi["capacity_weight"] = _safe_numeric(poi.get("capacity_weight", pd.Series(dtype=float)), default=1.0)
        poi["capacity_prior_band"] = poi["capacity_prior_band"].fillna(str(default_row["capacity_prior_band"])).astype(str)
        poi["capacity_band_scalar"] = poi["capacity_prior_band"].map(_capacity_band_scalar)
        poi["bug_prior_weight"] = (
            _safe_numeric(poi["bug_propensity_weight"]) *
            _safe_numeric(poi["night_use_weight"]) *
            _safe_numeric(poi["detectability_weight"])
        )
        poi["bug_capacity_weight"] = poi["bug_prior_weight"] * _safe_numeric(poi["capacity_weight"], default=1.0) * _safe_numeric(poi["capacity_band_scalar"], default=1.0)
        poi["bug_hours_weight"] = poi["bug_prior_weight"] * _safe_numeric(poi["night_use_weight"], default=0.25)
        poi["buffer_radius_m"] = np.where(poi["facility_type_std"] == "aerodrome", BUFFER_RADII["aerodrome"], DEFAULT_BUFFER)
        poi["high_conf_bug_flag"] = (
            (poi["bug_propensity_weight"] >= 0.8) &
            (poi["night_use_weight"] >= 0.8) &
            (poi["detectability_weight"] >= 0.65)
        ).astype(int)

        sub = out.loc[mask].copy()
# ... truncated for notebook readability (71 hidden lines) ...
```

### `project/modeling/pipelines/03_exploration_pipeline.py::run_bug_aware_transport`

```python
def run_bug_aware_transport(panel: pd.DataFrame, recovery: pd.DataFrame) -> SpecResult:
    specs: List[Tuple[str, pd.DataFrame, pd.DataFrame, List[str]]] = []

    specs.append(("BUG0", panel.copy(), recovery.copy(), BASE_NUMERIC + QUALITY_GUARD_NUMERIC))
    specs.append(("BUG1A", panel.copy(), recovery.copy(), BASE_NUMERIC + QUALITY_GUARD_NUMERIC + BUG_PRIOR_NUMERIC))

    panel_b = panel.copy()
    rec_b = recovery.copy()
    panel_b["legacy_in_buffer"] = panel_b["in_buffer"]
    panel_b["in_buffer"] = _safe_numeric(panel_b["high_conf_bug_buffer"]).astype(int)
    rec_b["legacy_in_buffer"] = rec_b["in_buffer"]
    rec_b["in_buffer"] = _safe_numeric(rec_b["high_conf_bug_buffer"]).astype(int)
    specs.append(("BUG1B", panel_b, rec_b, BASE_NUMERIC + QUALITY_GUARD_NUMERIC + BUG_PRIOR_NUMERIC))

    panel_c = panel.copy()
    rec_c = recovery.copy()
    panel_c["legacy_in_buffer"] = panel_c["in_buffer"]
    panel_c["in_buffer"] = _safe_numeric(panel_c["high_conf_bug_buffer"]).astype(int)
    rec_c["legacy_in_buffer"] = rec_c["in_buffer"]
    rec_c["in_buffer"] = _safe_numeric(rec_c["high_conf_bug_buffer"]).astype(int)
    specs.append(("BUG1C", panel_c, rec_c, QUALITY_GUARD_NUMERIC + BUG_PRIOR_NUMERIC))

    fold_parts: List[pd.DataFrame] = []
    agg_parts: List[pd.DataFrame] = []
    coef_parts: List[pd.DataFrame] = []
    for spec_id, spec_panel, spec_recovery, numeric_terms in specs:
        res = run_loeo_spec(
            spec_panel,
            spec_recovery,
            numeric_terms=numeric_terms,
            cat_terms=["land_use_group", "event_disaster_type"],
            experiment_family="bug_transport",
            spec_id=spec_id,
        )
        fold_parts.append(res.fold_df)
        agg_parts.append(res.agg_df)
        coef_parts.append(res.coef_df)

    fold_df = pd.concat(fold_parts, ignore_index=True) if fold_parts else pd.DataFrame()
    agg_df = pd.concat(agg_parts, ignore_index=True) if agg_parts else pd.DataFrame()
    coef_df = pd.concat(coef_parts, ignore_index=True) if coef_parts else pd.DataFrame()
    fold_df.to_csv(BUG_TRANSPORT_FOLD_PATH, index=False)
    agg_df.to_csv(BUG_TRANSPORT_AGG_PATH, index=False)
    return SpecResult(fold_df=fold_df, agg_df=agg_df, coef_df=coef_df)
```

### `project/modeling/pipelines/03_exploration_pipeline.py::attach_bug_detectability_features`

```python
def attach_bug_detectability_features(panel: pd.DataFrame, write_artifacts: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    out = panel.copy()
    events_cfg = load_json(CONFIG_EVENTS)
    lookup = _load_bug_detectability_lookup()
    default_row = {
        "bug_propensity_weight": 0.15,
        "night_use_weight": 0.25,
        "diesel_dominance_prior": 0.7,
        "detectability_weight": 0.25,
        "size_band_prior": "low",
        "hours_prior_band": "low",
    }
    if BUG_DETECT_CONFIG_PATH.exists():
        cfg = load_json(BUG_DETECT_CONFIG_PATH)
        if isinstance(cfg, dict) and isinstance(cfg.get("default"), dict):
            default_row.update(cfg["default"])

    feature_defaults = {
        "bug_detect_count_750m": 0.0,
        "bug_detect_count_1250m": 0.0,
        "bug_detect_capacity_proxy_1km": 0.0,
        "bug_detect_hours_proxy_1km": 0.0,
        "bug_detect_diesel_proxy_1km": 0.0,
        "bug_detect_score_1km": 0.0,
        "bug_detect_min_dist_m": np.nan,
        "high_detect_bug_buffer": 0,
    }
    for col, default in feature_defaults.items():
        if col not in out.columns:
            out[col] = default

    audit_rows: List[Dict[str, object]] = []

    for event_id, cfg in events_cfg.items():
        mask = out["event_id"].astype(str) == str(event_id)
        if mask.sum() == 0:
            continue

        poi_path = ROOT / str(cfg["poi_csv"])
        if not poi_path.exists():
            for col, default in feature_defaults.items():
                out.loc[mask, col] = default
            audit_rows.append(
                {
                    "event_id": event_id,
                    "n_obs": int(mask.sum()),
                    "poi_status": "missing_poi_csv",
                    "poi_count": 0,
                    "high_detect_poi_count": 0,
                    "high_detect_share": 0.0,
                    "bug_detect_count_750m_constant": 1,
                    "bug_detect_count_1250m_constant": 1,
                    "bug_detect_capacity_proxy_1km_constant": 1,
                    "bug_detect_hours_proxy_1km_constant": 1,
                    "bug_detect_diesel_proxy_1km_constant": 1,
                    "bug_detect_score_1km_constant": 1,
                    "high_detect_bug_buffer_share": 0.0,
                }
            )
            continue

        poi = pd.read_csv(poi_path).copy()
        if poi.empty:
            for col, default in feature_defaults.items():
                out.loc[mask, col] = default
            audit_rows.append(
                {
                    "event_id": event_id,
                    "n_obs": int(mask.sum()),
                    "poi_status": "empty_poi_csv",
                    "poi_count": 0,
                    "high_detect_poi_count": 0,
                    "high_detect_share": 0.0,
                    "bug_detect_count_750m_constant": 1,
                    "bug_detect_count_1250m_constant": 1,
                    "bug_detect_capacity_proxy_1km_constant": 1,
                    "bug_detect_hours_proxy_1km_constant": 1,
                    "bug_detect_diesel_proxy_1km_constant": 1,
                    "bug_detect_score_1km_constant": 1,
                    "high_detect_bug_buffer_share": 0.0,
                }
            )
            continue

        poi["facility_type_std"] = poi["type"].astype(str).map(standardize_facility_type)
        poi = poi.merge(lookup, on="facility_type_std", how="left")
        for c, default in [
            ("bug_propensity_weight", float(default_row["bug_propensity_weight"])),
            ("night_use_weight", float(default_row["night_use_weight"])),
            ("diesel_dominance_prior", float(default_row["diesel_dominance_prior"])),
            ("detectability_weight", float(default_row["detectability_weight"])),
        ]:
            poi[c] = _safe_numeric(poi[c], default=default)
        poi["size_band_prior"] = poi["size_band_prior"].fillna(str(default_row["size_band_prior"])).astype(str)
        poi["hours_prior_band"] = poi["hours_prior_band"].fillna(str(default_row["hours_prior_band"])).astype(str)
        poi["size_band_scalar"] = poi["size_band_prior"].map(_capacity_band_scalar)
        poi["hours_band_scalar"] = poi["hours_prior_band"].map(_hours_band_scalar)
        poi["bug_detect_weight"] = (
            _safe_numeric(poi["bug_propensity_weight"]) *
            _safe_numeric(poi["night_use_weight"]) *
            _safe_numeric(poi["diesel_dominance_prior"]) *
            _safe_numeric(poi["detectability_weight"])
        )
        poi["bug_detect_capacity_weight"] = poi["bug_detect_weight"] * _safe_numeric(poi["size_band_scalar"], default=1.0)
        poi["bug_detect_hours_weight"] = poi["bug_detect_weight"] * _safe_numeric(poi["hours_band_scalar"], default=1.0)
        poi["bug_detect_diesel_weight"] = poi["bug_detect_weight"] * _safe_numeric(poi["diesel_dominance_prior"], default=1.0)
        poi["bug_detect_score_weight"] = (
            poi["bug_detect_weight"] *
            _safe_numeric(poi["size_band_scalar"], default=1.0) *
            _safe_numeric(poi["hours_band_scalar"], default=1.0)
# ... truncated for notebook readability (87 hidden lines) ...
```

### `project/modeling/pipelines/03_exploration_pipeline.py::run_bug_detectability_transport`

```python
def run_bug_detectability_transport(panel: pd.DataFrame, recovery: pd.DataFrame) -> SpecResult:
    specs: List[Tuple[str, pd.DataFrame, pd.DataFrame, List[str]]] = []

    specs.append(("BD0", panel.copy(), recovery.copy(), BASE_NUMERIC + QUALITY_GUARD_NUMERIC))
    specs.append(("BD1A", panel.copy(), recovery.copy(), BASE_NUMERIC + QUALITY_GUARD_NUMERIC + BUG_DETECT_NUMERIC))

    panel_b = panel.copy()
    rec_b = recovery.copy()
    panel_b["legacy_in_buffer"] = panel_b["in_buffer"]
    panel_b["in_buffer"] = _safe_numeric(panel_b["high_detect_bug_buffer"]).astype(int)
    rec_b["legacy_in_buffer"] = rec_b["in_buffer"]
    rec_b["in_buffer"] = _safe_numeric(rec_b["high_detect_bug_buffer"]).astype(int)
    specs.append(("BD1B", panel_b, rec_b, BASE_NUMERIC + QUALITY_GUARD_NUMERIC + BUG_DETECT_NUMERIC))

    panel_c = panel.copy()
    rec_c = recovery.copy()
    panel_c["legacy_in_buffer"] = panel_c["in_buffer"]
    panel_c["in_buffer"] = _safe_numeric(panel_c["high_detect_bug_buffer"]).astype(int)
    rec_c["legacy_in_buffer"] = rec_c["in_buffer"]
    rec_c["in_buffer"] = _safe_numeric(rec_c["high_detect_bug_buffer"]).astype(int)
    specs.append(("BD1C", panel_c, rec_c, QUALITY_GUARD_NUMERIC + BUG_DETECT_NUMERIC))

    fold_parts: List[pd.DataFrame] = []
    agg_parts: List[pd.DataFrame] = []
    coef_parts: List[pd.DataFrame] = []
    for spec_id, spec_panel, spec_recovery, numeric_terms in specs:
        res = run_loeo_spec(
            spec_panel,
            spec_recovery,
            numeric_terms=numeric_terms,
            cat_terms=["land_use_group", "event_disaster_type"],
            experiment_family="bug_detectability_transport",
            spec_id=spec_id,
        )
        fold_parts.append(res.fold_df)
        agg_parts.append(res.agg_df)
        coef_parts.append(res.coef_df)

    fold_df = pd.concat(fold_parts, ignore_index=True) if fold_parts else pd.DataFrame()
    agg_df = pd.concat(agg_parts, ignore_index=True) if agg_parts else pd.DataFrame()
    coef_df = pd.concat(coef_parts, ignore_index=True) if coef_parts else pd.DataFrame()
    fold_df.to_csv(BUG_DETECT_FOLD_PATH, index=False)
    agg_df.to_csv(BUG_DETECT_AGG_PATH, index=False)
    return SpecResult(fold_df=fold_df, agg_df=agg_df, coef_df=coef_df)
```

## 4. BUG2 official-inventory validation

### `project/modeling/pipelines/03_exploration_pipeline.py::canonicalize_bug_inventory`

```python
def canonicalize_bug_inventory(df: pd.DataFrame, pilot_cfg: Dict[str, object]) -> pd.DataFrame:
    out = df.copy()
    rename_map = {
        "PTO_RECORD_ID": "record_id",
        "Concatenated Facility - Source": "record_id",
        "Facility #": "facility_id",
        "Source #": "source_id",
        "facility_type": "facility_type_raw",
        "facility_type_clean": "facility_type_std",
        "generator_type": "facility_type_raw",
        "EQUIPMENT_TYPE": "facility_type_raw",
        "PERMIT_DESCRIPTION": "facility_type_raw",
        "Primary Use": "facility_type_raw",
        "capacity": "capacity_kw",
        "kw": "capacity_kw",
        "Capacity listed (kW)": "capacity_kw",
        "Total Capacity (kW)": "capacity_kw",
        "Estimated Installed Capacity kW": "capacity_kw",
        "Estimated Utilized Capacity kW": "capacity_kw",
        "operating_hours": "operating_hours_annual",
        "hours": "operating_hours_annual",
        "LAST READING - FIRST READING": "operating_hours_annual",
        "Usage/yr": "operating_hours_annual",
        "address": "address_raw",
        "address_line": "address_raw",
        "EQUIP_LOCATION_ADDRESS": "address_raw",
        "Address_Full": "address_raw",
        "Plant Address": "address_raw",
        "latitude": "lat",
        "longitude": "lon",
        "Latitude": "lat",
        "Longitude": "lon",
        "DBA": "facility_name",
        "Regulated Entity Name": "facility_name",
        "FacilityName": "facility_name",
        "Fuel": "fuel_type",
        "FuelGIS": "fuel_type",
        "agency_url": "source_url",
        "Source": "source_url",
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})
    if "record_id" not in out.columns and {"facility_id", "source_id"}.issubset(out.columns):
        out["record_id"] = out["facility_id"].astype(str) + "-" + out["source_id"].astype(str)
    if "capacity_kw" in out.columns:
        out["capacity_kw"] = pd.to_numeric(out["capacity_kw"], errors="coerce")
    if ("capacity_kw" not in out.columns or out["capacity_kw"].isna().all()) and "Horsepower" in out.columns:
        hp = pd.to_numeric(out["Horsepower"], errors="coerce")
        out["capacity_kw"] = hp * 0.7457
    for col in _bug2_required_columns():
        if col not in out.columns:
            out[col] = np.nan
    out["source_dataset"] = out["source_dataset"].fillna(str(pilot_cfg.get("source_dataset", str(pilot_cfg.get("pilot_state", "PR")).lower() + "_pilot")))
    out["jurisdiction"] = out["jurisdiction"].fillna(str(pilot_cfg.get("jurisdiction", pilot_cfg.get("pilot_state", "PR"))))
    out["state"] = out["state"].fillna(str(pilot_cfg.get("pilot_state", "PR")))
    out["county_or_district"] = out["county_or_district"].fillna(str(pilot_cfg.get("county_or_district", "")))
    out["facility_type_raw"] = out["facility_type_raw"].fillna(out["facility_type_std"]).fillna("unknown").astype(str)
    mask_std = out["facility_type_std"].isna() | (out["facility_type_std"].astype(str).str.strip() == "")
    if mask_std.any():
        out.loc[mask_std, "facility_type_std"] = out.loc[mask_std, "facility_type_raw"].map(standardize_facility_type)
    out["facility_type_std"] = out["facility_type_std"].fillna("other").astype(str)
    out["record_id"] = out["record_id"].fillna(out.index.astype(str)).astype(str)
    out["facility_name"] = out["facility_name"].fillna("unknown_facility").astype(str)
    out["fuel_type"] = out["fuel_type"].fillna("unknown").astype(str)
    out["address_raw"] = out["address_raw"].fillna("").astype(str)
    out["source_url"] = out["source_url"].fillna(str(pilot_cfg.get("source_url", ""))).astype(str)
    out["capacity_kw"] = _safe_numeric(out["capacity_kw"])
    out["operating_hours_annual"] = _safe_numeric(out["operating_hours_annual"])
    out["lat"] = pd.to_numeric(out["lat"], errors="coerce")
    out["lon"] = pd.to_numeric(out["lon"], errors="coerce")
    if "Accuracy Type" in df.columns:
        acc_type = df["Accuracy Type"].fillna("unknown").astype(str)
    else:
        acc_type = pd.Series(["coords_present" if ok else "missing_coords" for ok in (np.isfinite(out["lat"]) & np.isfinite(out["lon"]))], index=out.index)
    if "Accuracy Score" in df.columns:
        acc_score = pd.to_numeric(df["Accuracy Score"], errors="coerce")
        geo_flag = np.where(
            np.isfinite(out["lat"]) & np.isfinite(out["lon"]),
            np.where(acc_score >= 0.8, "coords_high_conf", np.where(acc_score > 0, "coords_low_conf", acc_type)),
            "missing_coords",
        )
        out["geo_quality_flag"] = out["geo_quality_flag"].fillna(pd.Series(geo_flag, index=out.index))
    else:
        out["geo_quality_flag"] = out["geo_quality_flag"].fillna(np.where(np.isfinite(out["lat"]) & np.isfinite(out["lon"]), "coords_present", "missing_coords"))

    quality_flags: List[str] = []
    permit_status = df.get("PTO_STATUS", df.get("PO Type", pd.Series([""] * len(out), index=out.index))).fillna("").astype(str).str.lower()
    usage_raw = df.get("Usage/yr", pd.Series([np.nan] * len(out), index=out.index))
    usage_raw = pd.to_numeric(usage_raw, errors="coerce")
    usage_date = pd.to_datetime(df.get("Usage Date", pd.Series([np.nan] * len(out), index=out.index)), errors="coerce")
    latest_usage = usage_date.max()
    stale_cutoff = latest_usage - pd.DateOffset(years=5) if pd.notna(latest_usage) else pd.NaT
    for idx in out.index:
        flags: List[str] = []
        if (out.at[idx, "facility_type_std"] != "other") or (out.at[idx, "capacity_kw"] > 0) or (out.at[idx, "operating_hours_annual"] > 0):
            flags.append("usable")
        else:
            flags.append("sparse")
        if np.isfinite(out.at[idx, "operating_hours_annual"]) and float(out.at[idx, "operating_hours_annual"]) > 8760:
            flags.append("hours_outlier")
        if idx in usage_raw.index and pd.notna(usage_raw.at[idx]) and float(usage_raw.at[idx]) == -1.0:
            flags.append("confidential_usage")
        if idx in permit_status.index and permit_status.at[idx] and permit_status.at[idx] not in {"active", "permit", "pto", "issued"}:
            flags.append("inactive_permit")
        if pd.notna(stale_cutoff) and idx in usage_date.index and pd.notna(usage_date.at[idx]) and usage_date.at[idx] < stale_cutoff:
            flags.append("stale_usage")
        quality_flags.append("|".join(dict.fromkeys(flags)))
    out["attribute_quality_flag"] = out["attribute_quality_flag"].fillna(pd.Series(quality_flags, index=out.index)).astype(str)
    keep = _bug2_required_columns()
    return out[keep].copy()
```

### `project/modeling/pipelines/03_exploration_pipeline.py::audit_bug_inventory`

```python
def audit_bug_inventory(df: pd.DataFrame, pilot_cfg: Dict[str, object]) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    pilot_events = [str(v) for v in pilot_cfg.get("pilot_event_ids", [])]
    geo_cov = float((np.isfinite(df["lat"]) & np.isfinite(df["lon"])).mean()) if not df.empty else 0.0
    attr_cov = float(((df["capacity_kw"] > 0) | (df["operating_hours_annual"] > 0) | (df["facility_type_std"] != "other")).mean()) if not df.empty else 0.0
    rows.append(
        {
            "pilot_state": str(pilot_cfg.get("pilot_state", "PR")),
            "pilot_event_ids": ";".join(pilot_events),
            "records_n": int(df.shape[0]),
            "geo_coverage": geo_cov,
            "attribute_coverage": attr_cov,
            "capacity_nonzero_share": float((df["capacity_kw"] > 0).mean()) if not df.empty else 0.0,
            "hours_nonzero_share": float((df["operating_hours_annual"] > 0).mean()) if not df.empty else 0.0,
            "confidential_usage_share": float(df["attribute_quality_flag"].astype(str).str.contains("confidential_usage", na=False).mean()) if not df.empty else 0.0,
            "hours_outlier_share": float(df["attribute_quality_flag"].astype(str).str.contains("hours_outlier", na=False).mean()) if not df.empty else 0.0,
            "inactive_permit_share": float(df["attribute_quality_flag"].astype(str).str.contains("inactive_permit", na=False).mean()) if not df.empty else 0.0,
            "stale_usage_share": float(df["attribute_quality_flag"].astype(str).str.contains("stale_usage", na=False).mean()) if not df.empty else 0.0,
            "distinct_facility_types": int(df["facility_type_std"].astype(str).nunique()) if not df.empty else 0,
            "duplicate_record_share": float(df["record_id"].astype(str).duplicated().mean()) if not df.empty else 0.0,
            "gate_pass": int(
                (df.shape[0] >= int(pilot_cfg["coverage_gate"]["min_records"])) and
                (geo_cov >= float(pilot_cfg["coverage_gate"]["min_geo_coverage"]))
            ),
        }
    )
    out = pd.DataFrame(rows)
    out.to_csv(BUG2_QA_PATH, index=False)
    return out
```

### `project/modeling/pipelines/03_exploration_pipeline.py::attach_official_bug_features`

```python
def attach_official_bug_features(
    panel: pd.DataFrame,
    inventory: pd.DataFrame,
    pilot_cfg: Dict[str, object],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    out = panel.copy()
    feature_defaults = {
        "official_bug_count_1km": 0.0,
        "official_bug_kw_sum_1km": 0.0,
        "official_bug_hours_proxy_1km": 0.0,
        "official_bug_min_dist_m": np.nan,
        "official_bug_coverage_flag": 0,
    }
    for col, default in feature_defaults.items():
        if col not in out.columns:
            out[col] = default

    audit_rows: List[Dict[str, object]] = []
    events_cfg = load_json(CONFIG_EVENTS)
    pilot_events = {str(v) for v in pilot_cfg.get("pilot_event_ids", [])}
    inv = inventory[np.isfinite(inventory["lat"]) & np.isfinite(inventory["lon"])].copy()

    for event_id, cfg in events_cfg.items():
        mask = out["event_id"].astype(str) == str(event_id)
        if mask.sum() == 0:
            continue
        if event_id not in pilot_events or inv.empty:
            out.loc[mask, "official_bug_coverage_flag"] = 0
            audit_rows.append({"event_id": event_id, "inventory_records": 0, "coverage_flag": 0, "feature_nonzero_share": 0.0})
            continue

        transformer = Transformer.from_crs("EPSG:4326", str(cfg["metric_crs"]), always_xy=True)
        sub = out.loc[mask].copy()
        px, py = transformer.transform(sub["lon"].to_numpy(), sub["lat"].to_numpy())
        qx, qy = transformer.transform(inv["lon"].to_numpy(), inv["lat"].to_numpy())
        pix_xy = np.column_stack([px, py])
        inv_xy = np.column_stack([qx, qy])
        tree = cKDTree(inv_xy)
        idx_1000 = tree.query_ball_point(pix_xy, r=1000.0)
        kw = inv["capacity_kw"].to_numpy(dtype=float)
        hrs = inv["operating_hours_annual"].to_numpy(dtype=float)

        def _sum(ix: Sequence[int], arr: np.ndarray) -> float:
            if not ix:
                return 0.0
            return float(arr[np.asarray(ix, dtype=int)].sum())

        sub["official_bug_count_1km"] = [float(len(ix)) for ix in idx_1000]
        sub["official_bug_kw_sum_1km"] = [_sum(ix, kw) for ix in idx_1000]
        sub["official_bug_hours_proxy_1km"] = [_sum(ix, hrs) for ix in idx_1000]
        if len(inv_xy):
            dist_all, _ = tree.query(pix_xy, k=1)
            sub["official_bug_min_dist_m"] = dist_all.astype(float)
        else:
            sub["official_bug_min_dist_m"] = np.nan
        sub["official_bug_coverage_flag"] = 1

        out.loc[mask, "official_bug_count_1km"] = sub["official_bug_count_1km"].to_numpy(dtype=float)
        out.loc[mask, "official_bug_kw_sum_1km"] = sub["official_bug_kw_sum_1km"].to_numpy(dtype=float)
        out.loc[mask, "official_bug_hours_proxy_1km"] = sub["official_bug_hours_proxy_1km"].to_numpy(dtype=float)
        out.loc[mask, "official_bug_min_dist_m"] = sub["official_bug_min_dist_m"].to_numpy(dtype=float)
        out.loc[mask, "official_bug_coverage_flag"] = 1
        audit_rows.append(
            {
                "event_id": event_id,
                "inventory_records": int(inv.shape[0]),
                "coverage_flag": 1,
                "feature_nonzero_share": float((sub["official_bug_count_1km"] > 0).mean()),
            }
        )

    audit = pd.DataFrame(audit_rows)
    audit.to_csv(BUG2_FEATURE_AUDIT_PATH, index=False)
    return out, audit
```

## Why the BUG track changed direction

The `document/` folder is useful here even though it is not the main story source for the project:

- it documents why diesel back-up generators are a plausible mechanism worth testing
- it shows why capacity and operating-hour fields are messy in real inventories
- it motivates the shift from proxy-only tuning toward official inventory validation


## What the extension layer adds

- Quality-adjusted transport still tells the same story; `QT1` Logit `AUC = 0.4973`.
- Hazard-aware transport is the strongest appendix ranking line; `HZ1` Logit `AUC = 0.6025`.
- BUG proxy refinement is negative; `BUG0` Logit `AUC = 0.4973`, `BUG1A` Logit `AUC = 0.4965`.
- Detectability-aware BUG features are also negative; `BD0` Logit `AUC = 0.4973`, `BD1A` Logit `AUC = 0.4931`.
- Event readiness is now explicit; top score is `ian_charlotteharbor` with `94`.
- BUG2 Puerto Rico is still a data-acquisition track; `records_n = 0`, `gate_pass = 0`.


## Figures to read

### Exploration V2 summary
![](../figures/exploration_v2/exploration_auc_compare.png)

### Quality-adjusted transport comparison
![](../figures/exploration_v2/quality_matched_compare_v1.png)

### Hazard-aware transport comparison
![](../figures/exploration_v2/hazard_transport_compare_v1.png)

### BUG proxy transport comparison
![](../figures/exploration_v2/bug_transport_compare_v1.png)

### BUG detectability transport comparison
![](../figures/exploration_v2/bug_detectability_transport_compare_v1.png)

### Event readiness score overview
![](../figures/intl_stage_repair_v1/event_readiness_score_v1.png)


## 5. What this notebook should leave you with

- `03_exploration_pipeline.py` is not random extra code; it is the project's extension and falsification layer.
- Hazard-aware transport is the strongest appendix line.
- Better BUG proxies were useful to test, but they did not become a breakthrough.
- BUG2 is a mechanism-validation and data-acquisition track, not yet a new model winner.
